In [0]:
from pyspark.sql import functions as F

from delta.tables import DeltaTable

# ---------------- CONFIG ----------------

BRONZE_JIRA = "workspace.slainte_bronze.jira_issues_bronze"

GOLD_DB     = "workspace.slainte_gold"

DIM_TEAM    = f"{GOLD_DB}.dim_team"

# ---------------- SETUP ----------------

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

# ---------------- BUILD DIM ----------------

df_dim_team = (

    spark.table(BRONZE_JIRA)

    .select(

        F.col("user_id").alias("source_user_id"),

        F.col("project_key").alias("project"),

        F.col("support_group").alias("team_name")

    )

    .filter(

        F.col("user_id").isNotNull() &

        F.col("project_key").isNotNull() &

        F.col("support_group").isNotNull()

    )

    .distinct()

    .withColumn("team_id", F.monotonically_increasing_id() + 1)

    .select("team_id", "source_user_id", "project", "team_name")

)

# ---------------- WRITE (MERGE) ----------------

if not spark.catalog.tableExists(DIM_TEAM):

    df_dim_team.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(DIM_TEAM)

else:

    target = DeltaTable.forName(spark, DIM_TEAM)

    (

        target.alias("t")

        .merge(

            df_dim_team.alias("s"),

            "t.source_user_id = s.source_user_id AND "

            "t.project = s.project AND "

            "t.team_name = s.team_name"

        )

        .whenMatchedUpdateAll()

        .whenNotMatchedInsertAll()

        .execute()

    )

print("✅ dim_team merged successfully")
 